# Serialization Deep Dive — pickle, joblib, and ONNX

Before you can serve a model, you need to persist it to disk. The format you choose affects file size, loading speed, cross-platform portability, and inference performance. This notebook benchmarks three common formats side by side.

## Learning Objectives

By the end of this notebook you will be able to:
1. Save and load a model using `pickle` and compare with `joblib`
2. Export a sklearn model to ONNX using `skl2onnx` and run inference with `onnxruntime`
3. Benchmark inference speed across all three formats
4. Choose the right format based on deployment target and requirements

## 1. Install ONNX Dependencies

In [1]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "skl2onnx", "onnxruntime", "-q"],
    check=True
)
print("Dependencies installed")

Dependencies installed



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. Train the Model

In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data.astype(np.float32), iris.target,
    test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Test accuracy: {acc:.2%}")
print(f"Training samples: {X_train.shape[0]}, features: {X_train.shape[1]}")

Test accuracy: 100.00%
Training samples: 120, features: 4


## 3. Save and Load with pickle

`pickle` is Python's built-in serialization module. It can serialize almost any Python object, but the resulting file is Python-version-specific and contains no portability guarantees.

In [3]:
import pickle, time, os

PICKLE_PATH = "/tmp/iris_model.pkl"

# Save
t0 = time.perf_counter()
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(clf, f)
save_time = time.perf_counter() - t0

file_size_kb = os.path.getsize(PICKLE_PATH) / 1024
print(f"pickle save: {save_time*1000:.1f} ms, size: {file_size_kb:.1f} KB")

# Load
t0 = time.perf_counter()
with open(PICKLE_PATH, "rb") as f:
    clf_pkl = pickle.load(f)
load_time = time.perf_counter() - t0
print(f"pickle load: {load_time*1000:.1f} ms")

# Verify predictions match
pkl_acc = accuracy_score(y_test, clf_pkl.predict(X_test))
print(f"Loaded model accuracy: {pkl_acc:.2%}")
assert pkl_acc == acc, "Predictions diverged after pickle round-trip!"
print("Predictions match original: True")

pickle save: 1.0 ms, size: 170.6 KB
pickle load: 0.8 ms
Loaded model accuracy: 100.00%
Predictions match original: True


## 4. Save and Load with joblib

`joblib` uses memory-mapped arrays for large numpy objects — it's significantly faster and produces smaller files for models with many large arrays (like RandomForest's decision trees). For models dominated by numpy arrays, joblib is the standard choice.

In [4]:
import joblib

JOBLIB_PATH = "/tmp/iris_model.joblib"

# Save
t0 = time.perf_counter()
joblib.dump(clf, JOBLIB_PATH)
joblib_save_time = time.perf_counter() - t0

joblib_size_kb = os.path.getsize(JOBLIB_PATH) / 1024
print(f"joblib save: {joblib_save_time*1000:.1f} ms, size: {joblib_size_kb:.1f} KB")

# Load
t0 = time.perf_counter()
clf_jbl = joblib.load(JOBLIB_PATH)
joblib_load_time = time.perf_counter() - t0
print(f"joblib load: {joblib_load_time*1000:.1f} ms")

jbl_acc = accuracy_score(y_test, clf_jbl.predict(X_test))
print(f"Loaded model accuracy: {jbl_acc:.2%}")

# Compare
print(f"\nSize comparison:  pickle={file_size_kb:.1f} KB  vs  joblib={joblib_size_kb:.1f} KB")
print(f"Load comparison:  pickle={load_time*1000:.1f} ms  vs  joblib={joblib_load_time*1000:.1f} ms")

joblib save: 10.2 ms, size: 182.5 KB
joblib load: 5.0 ms
Loaded model accuracy: 100.00%

Size comparison:  pickle=170.6 KB  vs  joblib=182.5 KB
Load comparison:  pickle=0.8 ms  vs  joblib=5.0 ms


## 5. Export to ONNX

ONNX (Open Neural Network Exchange) is a cross-platform format. An ONNX model can run on Python, C++, Java, mobile, and GPU runtimes — all from the same file. `skl2onnx` converts sklearn models to ONNX; `onnxruntime` runs inference.

In [5]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ONNX_PATH = "/tmp/iris_model.onnx"

# Define input shape: batch_size=None (dynamic), 4 features
initial_type = [("float_input", FloatTensorType([None, 4]))]

# Convert
t0 = time.perf_counter()
onnx_model = convert_sklearn(clf, initial_types=initial_type)
convert_time = time.perf_counter() - t0

with open(ONNX_PATH, "wb") as f:
    f.write(onnx_model.SerializeToString())

onnx_size_kb = os.path.getsize(ONNX_PATH) / 1024
print(f"ONNX export: {convert_time*1000:.1f} ms, size: {onnx_size_kb:.1f} KB")

ONNX export: 7.3 ms, size: 78.3 KB


## 6. Run Inference with ONNX Runtime

In [6]:
import onnxruntime as ort

sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Show the ONNX graph's input and output names
print("ONNX graph inputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: shape={inp.shape}, type={inp.type}")

print("ONNX graph outputs:")
for out in sess.get_outputs():
    print(f"  {out.name}: shape={out.shape}, type={out.type}")

# Run inference
input_name = sess.get_inputs()[0].name
onnx_preds = sess.run(None, {input_name: X_test})
# onnx_preds[0] = predicted class, onnx_preds[1] = probabilities dict
onnx_classes = onnx_preds[0]

onnx_acc = accuracy_score(y_test, onnx_classes)
print(f"\nONNX inference accuracy: {onnx_acc:.2%}")

ONNX graph inputs:
  float_input: shape=[None, 4], type=tensor(float)
ONNX graph outputs:
  output_label: shape=[None], type=tensor(int64)
  output_probability: shape=[], type=seq(map(int64,tensor(float)))

ONNX inference accuracy: 100.00%


## 7. Benchmark — Inference Speed Across All Three Formats

In [7]:
N_RUNS = 1000
test_input = X_test  # 30 samples

# pickle model inference
t0 = time.perf_counter()
for _ in range(N_RUNS):
    clf_pkl.predict(test_input)
pkl_time = (time.perf_counter() - t0) / N_RUNS * 1000

# joblib model inference (same sklearn model, same speed)
t0 = time.perf_counter()
for _ in range(N_RUNS):
    clf_jbl.predict(test_input)
jbl_time = (time.perf_counter() - t0) / N_RUNS * 1000

# ONNX Runtime inference
t0 = time.perf_counter()
for _ in range(N_RUNS):
    sess.run(None, {input_name: test_input})
onnx_time = (time.perf_counter() - t0) / N_RUNS * 1000

print(f"Inference benchmark ({N_RUNS} runs, {len(test_input)} samples each):")
print(f"  pickle  model: {pkl_time:.3f} ms/call")
print(f"  joblib  model: {jbl_time:.3f} ms/call")
print(f"  ONNX Runtime:  {onnx_time:.3f} ms/call")

if pkl_time > 0:
    ratio = pkl_time / onnx_time
    faster = ratio >= 1.0
    print(f"\nONNX vs sklearn on THIS model: {ratio:.2f}x "
          f"({'faster' if faster else 'slower'})")
    if ratio >= 1.5:
        print("ONNX Runtime is faster here: it is a compiled inference engine, so")
        print("each call skips the Python/numpy dispatch overhead that sklearn's")
        print(".predict() pays on every call. That overhead gap is enough to speed")
        print("up even this tiny model.")
    elif ratio >= 0.9:
        print("Roughly the same speed here: on a model this small the fixed per-call")
        print("overhead dominates either way, so neither format clearly wins.")
    else:
        print("ONNX is slower here: its session/setup overhead can outweigh the")
        print("benefits on a model this small with tiny batches.")
    print("\nHow big the win is depends on model size, batch size, and hardware.")
    print("The classic '2-10x' figure is just a rough guide - the largest gains")
    print("show up on LARGE models, batched inputs, and GPU-enabled ONNX Runtime,")
    print("so your exact ratio will vary by machine.")


Inference benchmark (1000 runs, 30 samples each):
  pickle  model: 2.072 ms/call
  joblib  model: 1.955 ms/call
  ONNX Runtime:  0.024 ms/call

ONNX vs sklearn on THIS model: 86.68x (faster)
ONNX Runtime is faster here: it is a compiled inference engine, so
each call skips the Python/numpy dispatch overhead that sklearn's
.predict() pays on every call. That overhead gap is enough to speed
up even this tiny model.

How big the win is depends on model size, batch size, and hardware.
The classic '2-10x' figure is just a rough guide - the largest gains
show up on LARGE models, batched inputs, and GPU-enabled ONNX Runtime,
so your exact ratio will vary by machine.


## 8. Format Comparison Table

| Format | Cross-platform | File size (this model) | Inference speed | Best use case |
|---|---|---|---|---|
| pickle | Python only | medium | baseline | Quick experiments, same-Python-version deployment |
| joblib | Python only | smaller (large arrays) | same as pickle | sklearn + numpy models in Python serving |
| ONNX | Any language / hardware | varies | often faster (compiled engine); biggest gains at scale / on GPU | Production serving, mobile, C++/Java services |

> **Reading the benchmark above:** ONNX Runtime is a compiled engine, so it often runs *faster* than sklearn's `predict()` even on this toy model — it avoids the Python dispatch overhead paid on every `.predict()` call. Don't over-read one ratio, though: the magnitude depends on model size, batching, and hardware, and the largest, most reliable wins come at scale or on GPU.

**When pickle fails:** a pickle file created with Python 3.11 may fail to load on Python 3.8 because the protocol includes Python-version-specific class definitions. ONNX has no such constraint.


## 9. Summary

In this notebook you:
- Saved and loaded the same model with `pickle` and `joblib`, verifying predictions round-trip correctly
- Converted the sklearn model to ONNX using `skl2onnx` and ran inference with `onnxruntime`
- Benchmarked inference latency across all three formats on 1000 calls
- Learned when each format is appropriate: pickle/joblib for Python-only serving, ONNX for cross-platform or high-performance deployments

**The key trade-off:** pickle/joblib are the easiest to use; ONNX is harder to set up but portable and often faster.

## Self-Check (answer before scrolling up)

1. **Why can't you reliably load a pickle file created on Python 3.11 in Python 3.8?** What does ONNX do differently that avoids this problem?
2. **What does ONNX Runtime do that makes inference faster than calling sklearn's `predict()`?** Think about what happens at the compiler level.
3. **Which format would you use for deploying a model to a mobile app?** Why can't you use joblib there?